In [18]:
import csv
import networkx as nx


def csv_to_networkx(
    edges_csv=None,
    nodes_csv=None,
    directed=False,
    node_id_col="ID",
    node_direction_col="direction",
    edge_source_col="Source Node",
    edge_target_col="Target Node",
    edge_direction_col="relation",
):
    """
    Convert CSV files into a NetworkX graph with attributes.
    """

    graph = nx.Graph()

    # Load nodes (optional)
    if nodes_csv is not None:
        with open(nodes_csv, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                node_id = row[node_id_col]
                direction = row.get(node_direction_col)

                graph.add_node(node_id,direction=None if direction in (None, "") else direction)

    # Load edges (optional)
    if edges_csv is not None:
        with open(edges_csv, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                source = row[edge_source_col]
                target = row[edge_target_col]
                relation = row.get(edge_direction_col)
    
                if relation is None or relation == "":
                    edge_direction = None
                else:
                    edge_direction = relation
                
                graph.add_edge(source, target, direction=edge_direction)
            
    return graph


In [76]:
def build_layers(graph_A, graph_B):
    layers = {}  # final result dictionary
    print(len(graph_B.nodes()))
    print("Graph A Node:", len(graph_A.nodes()))
    
    temp = 0
    #print("Graph B Node:", temp)

    # Level containers
    level0 = []
    level1 = []

    # Identify level 1 nodes (common nodes in A and B)
    for b in graph_B.nodes():
        if b in graph_A:
            level1.append(b)
            temp = temp + 1
    print("Level 1 Nodes: ",temp)
    print("Graph A Node:", len(graph_B.nodes())-temp)
    
    # Build level 0 nodes (neighbors of level 1 in graph B)
    for b in level1:
        for node in graph_B.neighbors(b):
            if node not in level1 and node not in level0:
                level0.append(node)
                #temp += 1

    print(temp)
    layers[0] = level0

    # Higher layers using BFS on graph A
    visited = set(level0)
    queue = [(n, 0) for n in level0]  # (node, layer)

    while queue:
        node, d = queue.pop(0)

        for nbr in graph_A.neighbors(node):
            if nbr in visited:
                continue

            visited.add(nbr)
            layers.setdefault(d + 1, []).append(nbr)
            queue.append((nbr, d + 1))

    return layers


In [58]:
import networkx as nx


def filtering(GA, layers, puc_delete_threshold=0.2):
    """
    GA: nx.Graph - Graph A that is being processed
    layers: dict[int, list] - nodes of A split into layers
    OPTIONAL: puc_delete_threshold=0.4 = give threshold, otherwise default = 0.4
    ---------------------------------------------------------------------------
    Returns:
    H : nx.Graph - Final version of graph A
    """

    # copy graph H
    H = GA.copy()

    node_layer = {n: d for d, nodes in layers.items() for n in nodes}
#---------------------------------------------------------------------------------
    def get_rid_minority_edges(H, v, d, majority):
        #remove minority edges for v considering neighbors in previous layer (d-1) and same layer (d)
        
        for u in list(H.neighbors(v)):
            # consider previous layer (d-1) and same layer (d)
            if node_layer.get(u) not in (d - 1, d):
                continue

            rel = H[u][v].get("relation")
            dir_u = H.nodes[u].get("direction")

            if rel not in (+1, -1) or dir_u not in (+1, -1):
                continue

            vote = dir_u / rel  # +1 or -1
            # delete node if puc > threshold
            if vote != majority:
                H.remove_edge(u, v)
#-----------------------------------------------------------------------------------
    for d in sorted(layers):
        if d == 0:
            continue

        for v in layers[d]:
            if not H.has_node(v):
                continue

            pos_votes = neg_votes = total = 0

            # Consider only edges from previous layer (d-1) for current node
            for u in H.neighbors(v):
                # For each layer d>0, each node v votes based ONLY on neighbors in layer (d-1)
                if node_layer.get(u) != d - 1:
                    continue

                rel = H[u][v].get("relation")
                dir_u = H.nodes[u].get("direction")

                # only consider edges and nodes that are +1 or -1 (ignores 0's)
                if rel not in (+1, -1) or dir_u not in (+1, -1):
                    continue

                total += 1
                vote = dir_u / rel # vote = dir_u / rel
                if vote > 0:
                    pos_votes += 1
                else:
                    neg_votes += 1

            # If no nodes from previous layer, leave it unchanged
            if total == 0:
                continue

            # decide new node direction based on node and calculate puc_value
            #set v.direction and v.puc_value based on majority and minority fraction
            if pos_votes >= neg_votes:
                majority = +1
                puc = neg_votes / total
                H.nodes[v]["direction"] = +1
                H.nodes[v]["puc_value"] = puc

            elif neg_votes > pos_votes:
                majority = -1
                puc = pos_votes / total
                H.nodes[v]["direction"] = -1
                H.nodes[v]["puc_value"] = puc

            else:
                H.nodes[v]["direction"] = 0
                H.nodes[v]["puc_value"] = 0.5
                continue
                
            # If puc_value is low, delete nodes
            if puc > puc_delete_threshold:
                H.remove_node(v)
                continue

            # Get rid of edges accordingly
            get_rid_minority_edges(H, v, d, majority)

    return H
    # Return H after filtering


In [84]:
import pandas as pd
def main(B_nodes_csv,AA_edges_csv,AB_edges_csv,puc_delete_threshold=0.2):

    """
    graph_A = 
    read cpx_node.csv file
    takes two csv files new_pls-pls_edges.csv and new_pls-cpx_edges.csv. These are the edge files. 
    Each row in the first two columns represents an edge, with the first column as the source and target (though order doesn't matter
    because it's undirected graph. Let the third column be the relation attribute of the edge
    For every node (so all rows in the first column of new_pls-cpx_edges.csv), find the corresponding row in the first column of
    cpx_node.csv. The second column of that row is the node's direction attribute. Set that node's "direction" attribute to that value.
    graph A should be a NetworkX graph. All the nodes found in the first column of new_pls-cpx_edges.csv are level 0 nodes. Level 1 nodes
    would be all nodes that are a neighbor with a distance of 1 from level 0 nodes in Graph A. and level 2 a distance of 2, etc.

    """
    graph_A = csv_to_networkx(
        edges_csv="new_pls-pls_edges.csv",
        nodes_csv=None,
        directed=False,
        node_id_col=None,
        node_direction_col=None,
        edge_source_col="Metabolite 1",
        edge_target_col="Metabolite 2",
        edge_direction_col="Sign",
    )
    graph_A = csv_to_networkx(
        edges_csv="new_pls-cpx_edges.csv",
        nodes_csv="cpx_node.csv",
        directed=False,
        node_id_col="ID",
        node_direction_col="Mean Log2 Fold Change Direction (DSS)",
        edge_source_col="Gene",
        edge_target_col="Metabolite",
        edge_direction_col="Sign",
    )


    graph_B = csv_to_networkx(
        edges_csv="new_pls-cpx_edges.csv",
        nodes_csv="cpx_node.csv",
        directed=False,
        node_id_col="ID",
        node_direction_col="Mean Log2 Fold Change Direction (DSS)",
        edge_source_col="Gene",
        edge_target_col="Metabolite",
        edge_direction_col="Sign",
    )
    #for u, v in graph_B.nodes(data=True):
        #print(u, v)

    #for u, v, attrs in graph_B.edges(data=True):
        #print(u, v, attrs)

    
        


    #print(G_A.edges(data=True))


    # Build layers
    layers = build_layers(graph_A, graph_B)
    
    print("\nLayer counts:")
    for layer in sorted(layers.keys()):
        print(f"Layer {layer}: {len(layers[layer])}")

    #print(layers, "\n")
    #print(len(layers))
    # Run filtering
    H = filtering(graph_A, layers, puc_delete_threshold=puc_delete_threshold)
    
    #for node, attrs in H.nodes(data=True):
    #    print(node, attrs)

    #write_nodes_and_direction_to_csv(H, "H_nodes_direction.csv")
    
    #return H


In [80]:
main("cpx_node.csv","new_pls-pls_edges.csv","new_pls-cpx_edges.csv")

1114
Graph A Node: 529
Level 1 Nodes:  311
Graph A Node: 803
311


NetworkXError: The node ENSMUSG00000027374 is not in the graph.

In [28]:
import csv

def write_nodes_and_direction_to_csv(H, output_csv):
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "direction"])  # header

        for node, data in H.nodes(data=True):
            writer.writerow([node, data.get("direction")])

